# 02 — Skeleton Extraction: ShuttleSet (GDINO-guided)

Extracts dual-player skeletons for ShuttleSet using **Grounding DINO + YOLOv8-Pose**.
GDINO acts as a player-specific detector (prompt: `"badminton player ."`) providing
bounding-box priors; YOLO keypoints are only kept for detections that pass **both**:
1. A geometric court-region filter (centre 76% of width, min box height 10% of frame)
2. IoU ≥ 0.25 vs a spatially-valid GDINO prior

This dual filter eliminates chair-umpire / ball-kid contamination without relying on
IoU alone (which can fail when GDINO itself hallucinates an umpire box).

**Runs locally** — frames already extracted to `datasets_preprocessing/shuttleset_frames/`.
All 40 matches available. Set `MVP_MODE = False` to process the full dataset.

**Output format:** `(2, T_rally, 34)` per-rally `.npy` files  
- `dim 0` = [x, y]  
- `dim 1` = T_rally frames spanning the full rally (frame_start → frame_end)  
- `dim 2` = 34 joints: nodes 0–16 = P0 (top-court / smaller Y), 17–33 = P1 (bottom-court)

**Saved to:** `datasets_preprocessing/shuttleset_skeletons_gdino/{match_id}/r{rally:04d}.npy`

**Steps:**
1. (Optional) Cleanup old GDINO skeletons
2. Target matches & batch audit
3. GDINO + spatial-filter model setup
4. Load shot records
5. GDINO-guided extraction loop
6. Contamination scan + Y-sort verification
7. Dataset loading test

In [1]:
import os, sys, zipfile

# ── Colab setup (skipped when running locally) ────────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_PATH    = '/content/drive/MyDrive/FineBadminton/baddiev2_colab.zip'
    PROJECT_PATH = '/content/Baddiev2'

    if not os.path.exists(PROJECT_PATH):
        print("Extracting project files...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print("Done.")
    else:
        print("Project already extracted.")

    os.chdir(os.path.join(PROJECT_PATH, 'notebooks'))
    sys.path.insert(0, PROJECT_PATH)
    print(f"CWD: {os.getcwd()}")

    # Install yt-dlp (yt-dlp is more maintained than youtube-dl)
    os.system("pip install -q yt-dlp")
else:
    print("Local run")

Local run


In [1]:
!pip install -q tqdm transformers timm pillow

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import cv2
import torch
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

from src.config import (
    PROJECT_ROOT,
    SS_FRAMES, SS_OUTPUTS, SS_SKELETONS_GDINO,
    get_config,
)

#cfg = get_config()
#HALF = cfg.data.shot_window // 2   # 8 frames before/after hit frame
#print(f"Shot window T={cfg.data.shot_window}, half={HALF}")
# #print(f"GDINO skeletons → {SS_SKELETONS_GDINO}")

## 0. (Optional) Cleanup

Delete previously extracted GDINO skeletons to force a full re-extraction.

In [ ]:
import shutil
FORCE_REEXTRACT = False  # ← set True to wipe and re-extract

if FORCE_REEXTRACT and SS_SKELETONS_GDINO.exists():
    existing = list(SS_SKELETONS_GDINO.rglob('*.npy'))
    for f in existing:
        f.unlink()
    print(f"Deleted {len(existing)} GDINO skeleton files")
else:
    n = len(list(SS_SKELETONS_GDINO.rglob('*.npy'))) if SS_SKELETONS_GDINO.exists() else 0
    print(f"Skipping cleanup — {n} existing GDINO skeletons retained")

## 1. Target Matches & Batch Audit

Set `TARGET_MATCH` to a specific match ID to jump straight to it, or leave `""` to
auto-queue the next unprocessed batch.  
`BATCH_SIZE` controls how many matches to queue in one run.

In [ ]:
## ── Frame download cell (COMMENTED OUT) ──────────────────────────────────────
## Frames are already extracted locally as JPG and uploaded to Google Drive.
## Use the "Unzip frames from Drive" cell below instead.
## ─────────────────────────────────────────────────────────────────────────────
#
# import csv, subprocess, shutil
#
# if IN_COLAB:
#     match_csv_path = PROJECT_ROOT / 'datasets' / 'ShuttleSet' / 'set' / 'match.csv'
#     url_map = {}
#     with open(match_csv_path) as f:
#         for row in csv.DictReader(f):
#             url_map[row['video']] = row['url']
#     print(f"Loaded {len(url_map)} YouTube URLs from match.csv")
#
#     from src.config import SS_OUTPUTS, SS_FRAMES as _SS_FRAMES
#     json_match_ids = [jf.stem for jf in sorted(SS_OUTPUTS.glob('*.json'))]
#     target_ids = json_match_ids[:MVP_MATCHES] if MVP_MODE else json_match_ids
#
#     _SS_FRAMES.mkdir(parents=True, exist_ok=True)
#     for match_id in target_ids:
#         frame_dir = _SS_FRAMES / match_id
#         if frame_dir.exists() and len(list(frame_dir.glob('*.jpg'))) > 100:
#             print(f"  [SKIP] {match_id[:50]} — frames already present")
#             continue
#         yt_url = url_map.get(match_id)
#         if not yt_url:
#             print(f"  [SKIP] {match_id[:50]} — no YouTube URL")
#             continue
#         # ... (download + ffmpeg extract) ...
# else:
#     print("Local run — skipping download")

print("Download cell skipped — frames loaded from Drive zips below.")

In [3]:
# ── Priority target matches (~20K shots, ~120K frames total) ─────────────────
# Women's singles (9 matches)
_WOMEN = [
    'Carolina_Marin_An_Se_Young_HSBC_BWF_WORLD_TOUR_FINALS_2020_QuarterFinals',
    'Carolina_Marin_An_Se_Young_TOYOTA_THAILAND_OPEN_2021_SemiFinals',
    'Carolina_Marin_Pornpawee_Chochuwong_HSBC_BWF_WORLD_TOUR_FINALS_2020_SemiFinals',
    'Carolina_Marin_Neslihan_Yigit_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
    'Carolina_Marin_Supanida_Katethong_YONEX_Thailand_Open_2021_QuarterFinals',
    'Mia_Blichfeldt_Busanan_Ongbamrungphan_YONEX_Thailand_Open_2021_QuarterFinals',
    'Evgeniya_Kosetskaya_Michelle_Li_HSBC_BWF_WORLD_TOUR_FINALS_2020_QuarterFinals',
    'Ratchanok_Intanon_Pusarla_V._Sindhu_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
    'Pusarla_V._Sindhu_Pornpawee_Chochuwong_HSBC_BWF_WORLD_TOUR_FINALS_2020_QuarterFinals',
]
# Men's singles (4 matches, ordered by shot count desc)
_MEN = [
    'Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_Finals',
    'Kento_MOMOTA_CHOU_Tien_Chen_Denmark_Open_2018_Finals',
    'Hans-Kristian_Solberg_Vittinghus_Lee_Cheuk_Yu_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
    'CHOU_Tien_Chen_Jonatan_CHRISTIE_Indonesia_Open_2019_Quarter-finals',
]
PRIORITY_MATCHES = _WOMEN + _MEN   # 13 matches

# ── Batch config ──────────────────────────────────────────────────────────────
BATCH_SIZE    = 3    # how many matches to queue per run (ignored if TARGET_MATCH is set)
TARGET_MATCH  = ''   # paste a specific match ID here to jump straight to it
                     # e.g. 'Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_Finals'

print(f"Priority list: {len(PRIORITY_MATCHES)} matches  |  batch={BATCH_SIZE}")
if TARGET_MATCH:
    print(f"Target override: {TARGET_MATCH}")

Priority list: 13 matches  |  batch=3


## 2. Grounding DINO Setup

Load GDINO-tiny and define per-frame extraction helpers.

**Pipeline per frame:**
1. GDINO detects `"badminton player ."` → bounding-box priors
2. Spatial pre-filter: discard boxes outside the centre 76% of the frame width,
   or too close to the top edge, or too small (< 10% of frame height) — eliminates
   chair-umpire and ball-kid hallucinations geometrically
3. YOLOv8-Pose detects all persons → keypoints
4. Keep YOLO persons that are *also* in the valid court region and have IoU ≥ 0.25
   vs a spatially-valid GDINO prior
5. Fallback: spatially-filtered top-2 YOLO if fewer than 2 GDINO-matched players
6. Y-sort the two kept players → P0 (top-court, smaller Y), P1 (bottom-court)

In [4]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image as _PIL
from ultralytics import YOLO

_DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_MODEL_ID     = 'IDEA-Research/grounding-dino-tiny'
_BOX_THRESH   = 0.30
_TXT_THRESH   = 0.25
_IOU_THRESH   = 0.25
_GDINO_PROMPT = 'badminton player .'

# ── Spatial filter thresholds (broadcast-angle assumption) ────────────────────
# Chair umpire sits at x ≈ 5–10% of width on the side. Players are always in
# the middle 75% of the frame. Tiny boxes (< 10% frame height) = ball kids /
# distant spectators.
_X_MIN_FRAC   = 0.12   # ignore left  12% of frame width
_X_MAX_FRAC   = 0.88   # ignore right 12% of frame width
_Y_MIN_FRAC   = 0.05   # ignore top   5%  (score overlay)
_MIN_H_FRAC   = 0.10   # minimum box height as fraction of frame height

print(f"Loading GDINO-tiny on {_DEVICE}...")
_gdino_proc  = AutoProcessor.from_pretrained(_MODEL_ID)
_gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(_MODEL_ID).to(_DEVICE)
_gdino_model.eval()
print("  GDINO ready")

_yolo = YOLO('yolov8s-pose.pt')
print("  YOLOv8s-pose ready")


# ── helpers ───────────────────────────────────────────────────────────────────

@torch.no_grad()
def _gdino_detect(bgr):
    """Run GDINO on a BGR frame → (boxes, scores) arrays in pixel coords."""
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H, W = bgr.shape[:2]
    inp  = _gdino_proc(
        images=_PIL.fromarray(rgb), text=_GDINO_PROMPT, return_tensors='pt'
    ).to(_DEVICE)
    out  = _gdino_model(**inp)
    res  = _gdino_proc.post_process_grounded_object_detection(
        outputs=out, input_ids=inp['input_ids'],
        threshold=_BOX_THRESH, text_threshold=_TXT_THRESH,
        target_sizes=[(H, W)]
    )[0]
    return res['boxes'].cpu().numpy(), res['scores'].cpu().numpy()


def _iou(b1, b2):
    """Intersection-over-Union of two [x1,y1,x2,y2] boxes."""
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter    = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union    = ((b1[2]-b1[0])*(b1[3]-b1[1]) +
                (b2[2]-b2[0])*(b2[3]-b2[1]) - inter + 1e-9)
    return inter / union


def _in_court_region(box, W, H):
    """True if box centre is in the valid court zone and the box is large enough."""
    cx    = (box[0] + box[2]) / 2
    cy    = (box[1] + box[3]) / 2
    box_h = box[3] - box[1]
    return (W * _X_MIN_FRAC <= cx <= W * _X_MAX_FRAC and
            cy >= H * _Y_MIN_FRAC and
            box_h >= H * _MIN_H_FRAC)


def _extract_gdino_frame(bgr):
    """
    Extract a (2, 34) skeleton from one BGR frame using GDINO+YOLO with
    spatial filtering to eliminate chair-umpire / ball-kid contamination.
    Returns zeros on complete detection failure.
    """
    H, W = bgr.shape[:2]
    g_boxes_all, _ = _gdino_detect(bgr)

    # Filter GDINO priors to court region — eliminates chair-umpire hallucinations
    g_boxes = np.array([b for b in g_boxes_all if _in_court_region(b, W, H)]) \
              if len(g_boxes_all) else np.zeros((0, 4))

    res = _yolo(bgr, verbose=False)[0]
    out = np.zeros((2, 34), dtype=np.float32)
    if res is None or res.keypoints is None or res.boxes is None:
        return out

    y_boxes = res.boxes.xyxy.cpu().numpy()   # (N, 4)
    y_cls   = res.boxes.cls.cpu().numpy()
    y_kpts  = res.keypoints.xy.cpu().numpy() # (N, 17, 2)
    y_confs = res.boxes.conf.cpu().numpy()

    # GDINO-filtered: keep YOLO persons that (a) are in court region AND
    # (b) overlap with a spatially-valid GDINO prior
    kept_kpts, kept_confs, kept_cy = [], [], []
    for i, (yb, yc, yconf) in enumerate(zip(y_boxes, y_cls, y_confs)):
        if int(yc) != 0:
            continue
        if not _in_court_region(yb, W, H):        # spatial pre-filter
            continue
        max_iou = max((_iou(yb, gb) for gb in g_boxes), default=0.0) if len(g_boxes) else 0.0
        if max_iou >= _IOU_THRESH:
            kept_kpts.append(y_kpts[i])
            kept_confs.append(float(yconf))
            kept_cy.append(float(y_kpts[i][:, 1].mean()))

    # Fallback: spatially-filtered top-2 YOLO (still obeys court region)
    if len(kept_kpts) < 2:
        valid_idx = [
            i for i, (yb, yc) in enumerate(zip(y_boxes, y_cls))
            if int(yc) == 0 and _in_court_region(yb, W, H)
        ]
        if len(valid_idx) >= 2:
            valid_idx.sort(key=lambda i: -y_confs[i])
            top2 = valid_idx[:2]
            kept_kpts  = [y_kpts[j]  for j in top2]
            kept_cy    = [float(y_kpts[j][:, 1].mean()) for j in top2]
            kept_confs = [float(y_confs[j]) for j in top2]
        else:
            return out   # genuinely can't find 2 players in valid region

    # If more than 2 passed the filter, keep the 2 most confident
    if len(kept_kpts) > 2:
        order      = np.argsort(kept_confs)[::-1][:2]
        kept_kpts  = [kept_kpts[j]  for j in order]
        kept_cy    = [kept_cy[j]    for j in order]

    # Y-sort: smaller Y centroid → P0 (top-court)
    ys = np.argsort(kept_cy)
    p0, p1 = kept_kpts[ys[0]], kept_kpts[ys[1]]
    out[0, :17] = p0[:, 0];  out[1, :17] = p0[:, 1]
    out[0, 17:] = p1[:, 0];  out[1, 17:] = p1[:, 1]
    return out


print("\nGDINO + YOLO helpers ready.")

Loading GDINO-tiny on cpu...


The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

  GDINO ready
  YOLOv8s-pose ready

GDINO + YOLO helpers ready.


## 3. Load Shot Records

In [5]:
# Load all shot records grouped by match_id
match_records = {}   # match_id → list of records
for json_file in sorted(SS_OUTPUTS.glob('*.json')):
    if json_file.name == 'pipeline_summary.json':
        continue
    with open(json_file) as f:
        records = json.load(f)
    if records:
        match_id = records[0].get('match_id', json_file.stem)
        match_records[match_id] = records

print(f"Matches with JSON records: {len(match_records)}")

Matches with JSON records: 44


In [6]:
# ── Audit: which priority matches are ready / done / missing frames ───────────
_has_frames    = lambda mid: (SS_FRAMES / mid).exists() and \
                             len(list((SS_FRAMES / mid).glob('*.jpg'))) > 0
_has_skeletons = lambda mid: (SS_SKELETONS_GDINO / mid).exists() and \
                             len(list((SS_SKELETONS_GDINO / mid).glob('*.npy'))) > 0

no_frames      = [m for m in PRIORITY_MATCHES if not _has_frames(m)]
done_matches   = [m for m in PRIORITY_MATCHES if _has_frames(m) and _has_skeletons(m)]
pending_matches = [m for m in PRIORITY_MATCHES if _has_frames(m) and not _has_skeletons(m)]

print(f"Priority: {len(PRIORITY_MATCHES)}  |  Done: {len(done_matches)}  "
      f"|  Pending: {len(pending_matches)}  |  No frames: {len(no_frames)}")

if no_frames:
    print(f"\n[WARN] Missing frames — extract with shuttleset_streaming_pipeline.py first:")
    for m in no_frames:
        print(f"  • {m}")

# ── Build the queue for this run ──────────────────────────────────────────────
if TARGET_MATCH:
    if not _has_frames(TARGET_MATCH):
        print(f"\n[ERROR] No frames for: {TARGET_MATCH}")
        match_ids = []
    elif TARGET_MATCH not in match_records:
        print(f"\n[ERROR] No JSON records for: {TARGET_MATCH}")
        match_ids = []
    else:
        match_ids = [TARGET_MATCH]
        print(f"\nTargeting: {TARGET_MATCH}")
else:
    match_ids = [m for m in pending_matches if m in match_records][:BATCH_SIZE]

print(f"\nQueued ({len(match_ids)} match(es)):")
for mid in match_ids:
    n_frames = len(list((SS_FRAMES / mid).glob('*.jpg')))
    n_shots  = len(match_records.get(mid, []))
    already  = len(list((SS_SKELETONS_GDINO / mid).glob('*.npy'))) \
               if (SS_SKELETONS_GDINO / mid).exists() else 0
    print(f"  • {mid[:65]}  ({n_shots} shots, {n_frames} frames, {already} rallies done)")

Priority: 13  |  Done: 0  |  Pending: 13  |  No frames: 0

Queued (3 match(es)):
  • Carolina_Marin_An_Se_Young_HSBC_BWF_WORLD_TOUR_FINALS_2020_Quarte  (814 shots, 6752 frames, 0 rallies done)
  • Carolina_Marin_An_Se_Young_TOYOTA_THAILAND_OPEN_2021_SemiFinals  (765 shots, 5064 frames, 0 rallies done)
  • Carolina_Marin_Pornpawee_Chochuwong_HSBC_BWF_WORLD_TOUR_FINALS_20  (577 shots, 4215 frames, 0 rallies done)


## 4. GDINO-Guided Extraction

For each rally:  
1. Collect `T_rally = frame_end - frame_start + 1` consecutive frames (forward-fill on missing frames)  
2. Run `_extract_gdino_frame` on each frame → `(2, 34)`  
3. Forward-fill on complete detection failures  
4. Save as `(2, T_rally, 34)` `.npy`  

Already-extracted rallies are skipped automatically.

In [ ]:
from collections import defaultdict

SS_SKELETONS_GDINO.mkdir(parents=True, exist_ok=True)

for match_id in tqdm(match_ids, desc='Matches'):
    match_frame_dir = SS_FRAMES / match_id
    match_skel_dir  = SS_SKELETONS_GDINO / match_id
    match_skel_dir.mkdir(parents=True, exist_ok=True)

    if not match_frame_dir.exists():
        print(f"  [SKIP] No frames: {match_id}")
        continue

    # Build frame_num → path index (jpg takes priority over png)
    frame_index = {}
    for ext in ('*.jpg', '*.png'):
        for fp in match_frame_dir.glob(ext):
            try:
                fnum = int(fp.stem.replace('frame_', ''))
                if fnum not in frame_index:
                    frame_index[fnum] = fp
            except ValueError:
                pass

    if not frame_index:
        print(f"  [SKIP] Empty frame dir: {match_id}")
        continue

    # Group records by rally integer
    rally_groups = defaultdict(list)
    for rec in match_records[match_id]:
        rally_groups[rec['rally']].append(rec)

    saved = skipped = 0

    for rally_int, records in tqdm(sorted(rally_groups.items()),
                                   desc=f'  {match_id[:35]}', leave=False):
        rally_file = match_skel_dir / f"r{rally_int:04d}.npy"

        if rally_file.exists():
            saved += 1
            continue

        # All shots in a rally share the same rally_frame_start / rally_frame_end
        frame_start = records[0].get('rally_frame_start')
        frame_end   = records[0].get('rally_frame_end')
        if frame_start is None or frame_end is None:
            skipped += 1
            continue

        T_rally  = frame_end - frame_start + 1
        skel_arr = np.zeros((2, T_rally, 34), dtype=np.float32)
        n_fallback = 0

        for t in range(T_rally):
            fn  = frame_start + t
            fp  = frame_index.get(fn)
            bgr = cv2.imread(str(fp)) if fp else None
            if bgr is None:
                if t > 0:
                    skel_arr[:, t, :] = skel_arr[:, t - 1, :]
                n_fallback += 1
                continue
            sf = _extract_gdino_frame(bgr)
            if sf.sum() == 0 and t > 0:
                sf = skel_arr[:, t - 1, :]
                n_fallback += 1
            skel_arr[:, t, :] = sf

        np.save(rally_file, skel_arr)
        saved += 1

    total = len(list(match_skel_dir.glob('*.npy')))
    print(f"  {match_id[:50]}: rallies saved={saved}, skipped={skipped}, total_npy={total}")

grand_total = len(list(SS_SKELETONS_GDINO.rglob('*.npy')))
print(f"\nTotal GDINO rally skeletons: {grand_total}")

Matches:  33%|███▎      | 1/3 [9:43:49<19:27:38, 35029.35s/it]

  Carolina_Marin_An_Se_Young_HSBC_BWF_WORLD_TOUR_FIN: rallies saved=40, skipped=0, total_npy=40


## 5. Contamination Scan + Y-Sort Verification

Checks for residual contamination (chair umpire detected as P0).
- `UMPIRE_X_THRESH` = 300px (≈15% of 1920px frame width — umpire sits near left edge)
- Y-sort: P0 mean Y should always be smaller (higher on screen) than P1

In [ ]:
UMPIRE_X_THRESH = 300   # pixels; umpire sits near left edge of 1920px frame

skel_files = sorted(SS_SKELETONS_GDINO.rglob('*.npy'))
print(f"GDINO skeletons found: {len(skel_files)}")

if not skel_files:
    print("  No skeletons yet — run extraction cell first.")
else:
    y_ok = y_fail = 0
    contaminated = 0
    p0x_vals = []

    for f in skel_files:
        skel = np.load(f)               # (2, T, 34)
        p0_y = skel[1, :, :17].mean()
        p1_y = skel[1, :, 17:].mean()
        p0_x = skel[0, :, :17].mean()
        p0x_vals.append(p0_x)

        if p0_y < p1_y:
            y_ok += 1
        else:
            y_fail += 1

        if p0_x < UMPIRE_X_THRESH:
            contaminated += 1

    total = y_ok + y_fail
    print(f"\nY-sort  : {y_ok}/{total} OK ({100*y_ok/total:.1f}%),  "
          f"{y_fail} FAIL")
    print(f"Umpire  : {contaminated}/{total} contaminated "
          f"({100*contaminated/total:.1f}%)  [thresh={UMPIRE_X_THRESH}px]")
    print(f"P0 mean-X  — median={np.median(p0x_vals):.0f}px, "
          f"min={np.min(p0x_vals):.0f}px, max={np.max(p0x_vals):.0f}px")

    # Histogram of P0 x centroid
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(p0x_vals, bins=60, color='#60a5fa', alpha=0.8)
    ax.axvline(UMPIRE_X_THRESH, color='#ef4444', lw=2,
               label=f'Umpire threshold ({UMPIRE_X_THRESH}px)')
    ax.set_xlabel('P0 mean X centroid (pixels)')
    ax.set_ylabel('Shots')
    ax.set_title('GDINO extraction — P0 X centroid distribution\n'
                 '(values < threshold indicate chair umpire contamination)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5b. Visual Verification

Overlay skeletons on actual frames for a random selection of shots.

In [ ]:
COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),
    (6,8),(8,10),(5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]
P_KP  = [(0,255,255), (255,0,255)]   # cyan / magenta (BGR)
P_LMB = [(0,150,150), (150,0,150)]

def _draw_player(img, skel_xy, p):
    off = p * 17
    xs, ys = skel_xy[0, off:off+17], skel_xy[1, off:off+17]
    for j1, j2 in COCO_EDGES:
        x1,y1,x2,y2 = int(xs[j1]),int(ys[j1]),int(xs[j2]),int(ys[j2])
        if x1>0 and y1>0 and x2>0 and y2>0:
            cv2.line(img,(x1,y1),(x2,y2),P_LMB[p],2)
    for k in range(17):
        x,y = int(xs[k]),int(ys[k])
        if x>0 or y>0:
            cv2.circle(img,(x,y),5,P_KP[p],-1)
    ok = (xs>0)&(ys>0)
    if ok.any():
        cx,cy = int(xs[ok].mean()),int(ys[ok].mean())
        cv2.putText(img,f'P{p}',(cx-10,cy-14),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,P_KP[p],2)

# Pick a few shots to visualise
SHOW_SHOTS = 6
all_shot_files = sorted(SS_SKELETONS_GDINO.rglob('*.npy'))

if not all_shot_files:
    print("No GDINO skeletons yet.")
else:
    # Spread evenly across extracted shots
    idxs = np.linspace(0, len(all_shot_files)-1, SHOW_SHOTS, dtype=int)
    sample_files = [all_shot_files[i] for i in idxs]

    fig, axes = plt.subplots(1, SHOW_SHOTS, figsize=(4*SHOW_SHOTS, 5))
    if SHOW_SHOTS == 1:
        axes = [axes]

    for ax, sf in zip(axes, sample_files):
        # Recover match_id / rally / ball from path
        ball_name = sf.stem                        # e.g. r0001_b0003
        match_id  = sf.parent.name
        parts     = ball_name.split('_')
        rally     = int(parts[0][1:])              # strip leading 'r'
        ball_rnd  = int(parts[1][1:])              # strip leading 'b'

        skel = np.load(sf)                         # (2, T, 34)
        t_mid = skel.shape[1] // 2

        # Find the corresponding frame on disk (centre of the shot window)
        # Read JSON to get frame_num
        json_file = next(iter(SS_OUTPUTS.glob(f'{match_id}*.json')), None)
        frame_num = None
        if json_file:
            with open(json_file) as jf:
                for rec in json.load(jf):
                    if rec.get('rally')==rally and rec.get('ball_round')==ball_rnd:
                        frame_num = rec.get('frame_num')
                        break

        img = None
        if frame_num is not None:
            fp = SS_FRAMES / match_id / f"frame_{frame_num:06d}.png"
            if fp.exists():
                img = cv2.imread(str(fp))

        if img is None:
            ax.text(0.5,0.5,'frame\nmissing',ha='center',va='center')
            ax.axis('off')
            continue

        for p in range(2):
            _draw_player(img, skel[:, t_mid, :], p)

        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{match_id[:20]}\nR{rally} B{ball_rnd} t={t_mid}', fontsize=7)
        ax.axis('off')

    plt.suptitle(
        'GDINO-guided extraction — centre frame of each shot\n'
        'Cyan = P0 (top-court, smaller Y)   Magenta = P1 (bottom-court)',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()

## 6. Dataset Loading Test

Verify `ShuttleSetDataset` can load GDINO skeletons through the standard pipeline.

In [ ]:
from src.data.dataset import ShuttleSetDataset

# Pass gdino=True (or skeleton_dir override) once ShuttleSetDataset supports it
# For now, verify the npy files load correctly with the expected shape
from src.config import SS_SKELETONS_GDINO
from src.data.feature_eng import FeatureEngineer

eng = FeatureEngineer(feature_layer='L2')

gdino_files = sorted(SS_SKELETONS_GDINO.rglob('*.npy'))
print(f"GDINO skeleton files: {len(gdino_files)}")

if gdino_files:
    sample = np.load(gdino_files[0])
    print(f"  Raw shape : {sample.shape}  (expected (2, {cfg.data.shot_window}, 34))")
    feats = eng.compute(sample)
    print(f"  L2 features shape: {feats.shape}  (expected (9, {cfg.data.shot_window}, 34))")
    print(f"  Non-zero elements: {(feats != 0).sum()} / {feats.size}")
    print(f"  P0 mean-Y: {sample[1, :, :17].mean():.1f}  P1 mean-Y: {sample[1, :, 17:].mean():.1f}")

# Standard dataset (still uses original skeletons dir — update when all matches extracted)
ss_ds = ShuttleSetDataset(feature_layer='L2')
print(f"\nShuttleSetDataset: {len(ss_ds)} samples")
if len(ss_ds) > 0:
    x, st = ss_ds[0]
    print(f"  Sample shape: {x.shape}, shot_type idx: {st}")

In [ ]:
## ── Save extracted skeletons to Google Drive (Colab only) ────────────────────
## Run this after extraction to persist results before the session ends.
## Saves to: MyDrive/FineBadminton/shuttleset_skeletons_gdino.zip

if IN_COLAB:
    import shutil, os
    from pathlib import Path

    DRIVE_DIR  = Path('/content/drive/MyDrive/FineBadminton')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    skel_dir   = SS_SKELETONS_GDINO
    out_zip    = DRIVE_DIR / 'shuttleset_skeletons_gdino.zip'

    n_files = len(list(skel_dir.rglob('*.npy'))) if skel_dir.exists() else 0
    print(f"Skeletons to archive: {n_files} .npy files from {skel_dir}")

    if n_files == 0:
        print("[WARN] No skeletons found — run extraction first.")
    else:
        print(f"Zipping to {out_zip} ...")
        shutil.make_archive(str(out_zip).replace('.zip', ''), 'zip', skel_dir.parent, skel_dir.name)
        size_mb = out_zip.stat().st_size / 1e6
        print(f"Saved: {out_zip} ({size_mb:.1f} MB, {n_files} skeletons)")
else:
    print("Local run — skeletons already on disk, no Drive save needed")